## This code is used to collect cutom dataset for poem generation
The data was collected from https://www.wishafriend.com/poems/.

In [ ]:
%pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import os
from urllib.parse import urljoin

BASE_URL = "https://www.wishafriend.com/poems/"
HEADERS = {
    'User-Agent': 'Mozilla/5.0'
}
OUTPUT_FILE = "poems.csv"
DELAY = 1 


def get_theme_links():
    """Get all theme category links from the main page"""
    try:
        response = requests.get(BASE_URL, headers=HEADERS)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        theme_links = []
        poem_divs = soup.find_all('div', id='div_poem')
        for div in poem_divs:
            for li in div.find_all('li'):
                a = li.find('a')
                if a and a.has_attr('href'):
                    full_url = urljoin(BASE_URL, a['href'])
                    theme_links.append((a.text.strip(), full_url))

        seen = set()
        unique_theme_links = []
        for theme in theme_links:
            if theme[1] not in seen:
                seen.add(theme[1])
                unique_theme_links.append(theme)

        return unique_theme_links
    except Exception as e:
        print(f"Error getting theme links: {e}")
        return []


def get_poems_from_theme(theme_name, theme_url, seen_poems=None):
    if seen_poems is None:
        seen_poems = set()

    poems = []
    visited_pages = set()

    def scrape_single_page(url):
        try:
            response = requests.get(url, headers=HEADERS)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            poem_divs = soup.find_all('div', class_='box col1')
            for div in poem_divs:
                p_tag = div.find(
                    'p', id=lambda x: x and x.startswith('msgcpy'))
                if not p_tag:
                    continue

                h4_tag = p_tag.find('h4')
                if not h4_tag:
                    continue

                title = h4_tag.text.strip()
                h4_tag.extract()
                body = p_tag.get_text(separator='\n').strip()

                key = (title, body)
                if key in seen_poems:
                    continue
                seen_poems.add(key)

                poems.append((theme_name, title, body))

            return soup
        except Exception as e:
            print(f"Error scraping page {url}: {e}")
            return None

    # Scrape first page
    soup = scrape_single_page(theme_url)
    if not soup:
        return poems

    visited_pages.add(theme_url)

    # Get pagination links
    pagination_links = soup.select(
        'div[style*="float:right"] a[href*="?pagenum="]')
    page_numbers = set()

    for a in pagination_links:
        href = a.get('href')
        if href and 'pagenum=' in href:
            try:
                num = int(href.split('pagenum=')[-1])
                page_numbers.add(num)
            except ValueError:
                continue

    # Scrape additional pages
    for page_num in sorted(page_numbers):
        page_url = f"{theme_url.split('?')[0]}?pagenum={page_num}"
        if page_url in visited_pages:
            continue
        time.sleep(DELAY)
        soup = scrape_single_page(page_url)
        visited_pages.add(page_url)

    return poems


def load_scraped_themes(filepath):
    """Load already scraped themes from the file"""
    scraped = set()
    if not os.path.exists(filepath):
        return scraped

    try:
        with open(filepath, 'r', encoding='utf-8') as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                scraped.add(row['Theme'])
    except Exception as e:
        print(f"Warning: Couldn't read existing file: {e}")

    return scraped


def append_to_csv(data, filename):
    """Append new poem data to CSV"""
    file_exists = os.path.isfile(filename)
    with open(filename, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        if not file_exists:
            writer.writerow(['Theme', 'Title', 'Poem'])
        writer.writerows(data)


def main():
    print("Starting poem scraping from wishafriend.com...\n")

    themes = get_theme_links()
    print(f"Found {len(themes)} themes.")

    already_scraped = load_scraped_themes(OUTPUT_FILE)
    print(f"{len(already_scraped)} themes already scraped.")

    seen_poems = set()

    for idx, (theme_name, theme_url) in enumerate(themes, 1):
        if theme_name in already_scraped:
            print(
                f"[{idx}/{len(themes)}] Skipping already scraped theme: {theme_name}")
            continue

        print(f"\n[{idx}/{len(themes)}] Scraping poems from '{theme_name}'...")
        print(f"URL: {theme_url}")
        time.sleep(DELAY)

        poems = get_poems_from_theme(theme_name, theme_url, seen_poems)
        append_to_csv(poems, OUTPUT_FILE)

        print(f"  Found and saved {len(poems)} poems.")

    print(f"\n✅ Done scraping! Poems saved in '{OUTPUT_FILE}'.")


if __name__ == "__main__":
    main()

Starting poem scraping from wishafriend.com...

Found 76 themes.
1 themes already scraped.
[1/76] Skipping already scraped theme: Love

[2/76] Scraping poems from 'I Love You'...
URL: https://www.wishafriend.com/love/i-love-you-poems.php
  Found and saved 62 poems.

[3/76] Scraping poems from 'Romantic'...
URL: https://www.wishafriend.com/love/romantic-poems.php
  Found and saved 72 poems.

[4/76] Scraping poems from 'Boyfriend'...
URL: https://www.wishafriend.com/love/poems-for-boyfriend.php
  Found and saved 76 poems.

[5/76] Scraping poems from 'Girlfriend'...
URL: https://www.wishafriend.com/love/poems-for-girlfriend.php
  Found and saved 76 poems.

[6/76] Scraping poems from 'Short Love Poems'...
URL: https://www.wishafriend.com/love/short-love-poems.php
  Found and saved 66 poems.

[7/76] Scraping poems from 'Friendship'...
URL: https://www.wishafriend.com/friendship/friendship-poems.php
  Found and saved 52 poems.

[8/76] Scraping poems from 'Friends'...
URL: https://www.wishafr

## Dataset checking

In [17]:
import pandas as pd
all_poems = pd.read_csv("poems.csv")
all_poems.head()

,Theme,Title,Poem
0,Love,Love-Struck For You,"My silent words shall speak,\n\nAs I see you w..."
1,Love,The Tale Of Love,"On my fading day, I may\n\nSneak in, to throw ..."
2,Love,An Angelic Love,"Smother me in your wings,\n\nFlap them to sorc..."
3,Love,Love At First Sight,"I fell deep in love, the first time I saw you,..."
4,Love,Bunch Of Happiness,Your smile shines brighter than even the morni...


In [18]:
len(all_poems)

2345